# Revenue Forecast SQL Notebook

Loads the parquet artefacts into DuckDB, defines a few views and runs a sample per-quarter forecast accuracy query.

After execution, the resulting DataFrames can be explored or written to CSV for Tableau.


In [1]:
import duckdb
import pandas as pd
from pathlib import Path

# Works whether the kernel's cwd is the project root or the notebooks/ folder
here = Path.cwd().resolve()
ROOT = here if (here / "data" / "processed").exists() else here.parent
print("project root:", ROOT)

# 1. Connection and views
conn = duckdb.connect()
conn.execute(f"""
    CREATE OR REPLACE VIEW vw_forecasts AS SELECT * FROM
        read_parquet('{(ROOT / "data/processed/backtest_results.parquet").as_posix()}');
    CREATE OR REPLACE VIEW vw_panel AS SELECT * FROM
        read_parquet('{(ROOT / "data/processed/revenue_quarterly.parquet").as_posix()}');
    CREATE OR REPLACE VIEW vw_sec AS SELECT ticker, tag, val, fy, form,
        try_cast(start as date) AS fy_start,
        try_cast("end" as date) AS fy_end,
        try_cast(filed as date) AS filed
    FROM read_parquet('{(ROOT / "data/raw/raw_all.parquet").as_posix()}')
    WHERE form = '10-K';
    CREATE OR REPLACE VIEW vw_rate AS
        SELECT try_cast(observation_date as date) AS d, value
        FROM read_parquet('{(ROOT / "data/raw/fred_MORTGAGE30US.parquet").as_posix()}');
""")

# 2. Sample query - per-quarter forecast accuracy
accuracy_sql = """
WITH w AS (
    SELECT ticker, ANY_VALUE(sector) AS sector, period_end AS quarter_end,
           MAX(actual) AS actual,
           MAX(CASE WHEN model='seasonal_drift' THEN forecast END) AS fc_chosen,
           MAX(CASE WHEN model='ridge_drivers' THEN forecast END) AS fc_ridge
    FROM vw_forecasts
    GROUP BY ticker, period_end
), a AS (
    SELECT *,
           round(100.0*(actual-fc_chosen)/fc_chosen,2) AS var_chosen_pct,
           round(100.0*(actual-fc_ridge)/fc_ridge,2) AS var_ridge_pct
    FROM w
)
SELECT * FROM a;
"""

df_accuracy = conn.execute(accuracy_sql).df()
print(df_accuracy.head())


project root: C:\Users\allyg\DataAnalysisProject\revenue-forecast-project
  ticker            sector quarter_end        actual     fc_chosen  \
0   BLDR  home_improvement  2014-12-31  3.967370e+08  3.984031e+08   
1   BLDR  home_improvement  2016-03-31  1.397114e+09  1.361360e+09   
2   BLDR  home_improvement  2016-09-30  1.745958e+09  4.637580e+09   
3   BLDR  home_improvement  2018-06-30  2.089888e+09  2.044539e+09   
4   BLDR  home_improvement  2018-09-30  2.118467e+09  2.130264e+09   

       fc_ridge  var_chosen_pct  var_ridge_pct  
0  3.837438e+08           -0.42           3.39  
1  1.004918e+09            2.63          39.03  
2  3.966969e+09          -62.35         -55.99  
3  2.031383e+09            2.22           2.88  
4  2.066362e+09           -0.55           2.52  
